<a href="https://colab.research.google.com/github/RPG-coder/newswise-bbc-kaggle/blob/main/NewsWise.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install kaggle

In [1]:
import os
from google.colab import userdata
os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
os.environ['KAGGLE_KEY'] = userdata.get('KAGGLE_KEY')
os.environ['ANTHROPIC_API_KEY'] = userdata.get('ANTHROPIC_API_KEY')

In [2]:
!kaggle datasets download hgultekin/bbcnewsarchive
!unzip bbcnewsarchive.zip

Dataset URL: https://www.kaggle.com/datasets/hgultekin/bbcnewsarchive
License(s): copyright-authors
100% 1.81M/1.81M [00:00<00:00, 154MB/s]

Archive:  bbcnewsarchive.zip
  inflating: bbc-news-data.csv       


In [3]:
import pandas as pd
df = pd.read_csv('bbc-news-data.csv', sep='\t',                 on_bad_lines='skip')

In [4]:
df.columns.tolist()

['category', 'filename', 'title', 'content']

In [5]:
df.head(2)

,category,filename,title,content
0,business,001.txt,Ad sales boost Time Warner profit,Quarterly profits at US media giant TimeWarne...
1,business,002.txt,Dollar gains on Greenspan speech,The dollar has hit its highest level against ...


In [6]:
df.shape

(2225, 4)

In [7]:
df['category'].value_counts()

,count
category,
sport,511
business,510
politics,417
tech,401
entertainment,386


In [8]:
!pip install langchain langchain-anthropic langchain-community langchain-huggingface faiss-gpu langgraph sentence-transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.9/63.9 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 54.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.3/135.3 MB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 58.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 50.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 572.1/572.1 kB 33.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 6.6 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 1.6.1
    Uninstalling langchain-core-1.6.1:
      Successfully uninstalled langchain-core-1.6.1
ERROR: pip's dependency res

In [9]:
from langchain_core.documents import Document

doc = [
    Document(
        page_content=row['content'],
        metadata  = {
            'title': row['title'],
            'category': row['category']
        }
    )
    for _, row in df.iterrows()
]

In [10]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

embedding = HuggingFaceEmbeddings(model_name='sentence-transformers/all-MiniLM-L6-v2')
vectorstore = FAISS.from_documents(doc, embedding)

/tmp/ipykernel_1401/323419157.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [11]:
retriever = vectorstore.as_retriever(search_kwargs={'k': 3})

In [87]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

history = []

prompt = ChatPromptTemplate([
    ('system', """You are a news research assistant.
Use ONLY the provided context to answer the question.
Do not use any external knowledge.
If you are not sure about the answer, just say that you don't know."""),
    MessagesPlaceholder(variable_name='history'),
    ('human', """Question: {question}
Context: {context}
Response:""")
])

In [88]:
from langchain_anthropic import ChatAnthropic
llm = ChatAnthropic(model="claude-sonnet-4-6")

In [89]:
from typing import TypedDict

class NewsState(TypedDict):
  question: str
  output: str


In [90]:
def ask_question(state: NewsState) -> NewsState:
  return {
      'question': input("Enter a Question: ")
  }

In [121]:
from langchain_core.runnables import RunnableLambda
from langchain_core.output_parsers import StrOutputParser
from langchain_core.messages import HumanMessage, AIMessage

def answer_question(state: NewsState) -> NewsState:
  chain = {
      'question': RunnableLambda(lambda s: s['question']),
      'context': RunnableLambda(lambda s: s['question']) | retriever,
      'history': RunnableLambda(lambda s: s['history']),
  } | prompt | llm | StrOutputParser()
  result = chain.invoke({
      'question': state['question'],
      'history': history
  })
  print('Agent:', result)
  history.append(HumanMessage(content=state['question']))
  history.append(AIMessage(content=result))
  return {
      'output': result
  }

In [122]:
def question_route(state: NewsState) -> str:
  if state['question'] == '/exit':
    return END
  return 'answer'

In [123]:
from langgraph.graph import StateGraph, START, END

builder = StateGraph(NewsState)
builder.add_node('ask', ask_question)
builder.add_node('answer', answer_question)

builder.add_edge(START, 'ask')
builder.add_conditional_edges('ask', question_route)
builder.add_edge('answer', 'ask')

In [124]:
from langgraph.checkpoint.memory import MemorySaver

saver = MemorySaver()
config = {'configurable': {'thread_id': 'newswise-1'}}

In [125]:
graph = builder.compile(checkpointer=saver)

In [126]:
result = graph.invoke({
    'question': '',
    'output': '',
}, config=config)
print(result)

Enter a Question: What happened in football?
Agent: Based on the provided context, here is what happened in football:

1. **Ferguson Urges Henry Punishment** - Sir Alex Ferguson called on the Football Association to punish Arsenal's Thierry Henry for an incident involving Gabriel Heinze. Ferguson believes Henry deliberately caught Heinze on the head with his knee during United's controversial win. He also criticized Arsenal's overall discipline, noting:
   - 24 fouls by Arsenal during the game
   - 7 fouls on Heinze
   - 5 fouls on Ronaldo
   - Vieira was only booked on his sixth foul, while Phil Neville was booked for his first challenge

2. **Middlesbrough's UEFA Cup Victory** - Middlesbrough beat Graz AK **2-1** in the UEFA Cup, booking their place in the last 16. However, the win was marred by an injury to **Stewart Downing**, who tweaked his hamstring and was carried off on a stretcher. Jimmy Floyd Hasselbaink scored, with his shot going through the goalkeeper's legs. Boro will ne

In [73]:
# from langchain_core.runnables import RunnablePassthrough
# from langchain_core.output_parsers import StrOutputParser

# With Langchain only
# chain = {
#     'question': RunnablePassthrough(),
#     'context': retriever
# } | prompt | llm | StrOutputParser()
# result = chain.invoke("What happened in football?")
# print(result)

Based on the provided context, several football-related events were reported:

1. **Rugby (Six Nations)**: England's Josh Lewsey disputed a disallowed try during England's loss to Ireland, claiming referee Jonathan Kaplan made a wrong decision. Teammate Mark Cueto also had a try disallowed in the same match.

2. **Premier League (Arsenal vs Manchester United)**: Sir Alex Ferguson called on the Football Association to punish Arsenal's Thierry Henry for an incident involving Gabriel Heinze, where Henry's knee collided with the back of Heinze's head. Ferguson also criticized Arsenal's overall discipline during the match, citing 24 fouls committed by Arsenal.

3. **UEFA Cup**: Middlesbrough defeated Graz AK **2-1**, booking their place in the last 16, though the victory was marred by a hamstring injury to winger Stewart Downing. Boro were set to face Sporting Lisbon in the next round.
